# Machine Learning Projektabschluss: Modellvergleich und Analyse

## Übersicht

Dieses Notebook lädt die trainierten und optimierten Modelle aus den Notebooks 02–05:

- **Notebook 02**: Decision Tree mit Cost Complexity Pruning
- **Notebook 03**: k-Nearest Neighbors mit PCA und Random Subspace
- **Notebook 04**: Feedforward Neural Network (FFNN) mit Regularisierung
- **Notebook 05**: Convolutional Neural Network (CNN) mit Dropout

Wir werden:
1. Alle gespeicherten Modelle laden
2. Ihre Leistungen vergleichen
3. Das beste Modell identifizieren
4. Eine ausführliche technische und visuelle Analyse liefern
5. Schlussfolgerungen und Empfehlungen ziehen


## 1. Setup: Imports und Pfade

In [ ]:
import os
import pathlib
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

import tensorflow as tf
from tensorflow.keras.models import load_model
from PIL import Image
from tqdm import tqdm

# Seed für Reproduzierbarkeit
np.random.seed(42)
tf.random.set_seed(42)

# Pfade (relativ zum notebooks-Ordner)
repo_root = pathlib.Path().resolve().parent
data_dir = repo_root / "data"
models_dir = repo_root / "models"
splits_csv = data_dir / "splits"
features_dir = data_dir / "features"

models_dir.mkdir(parents=True, exist_ok=True)
splits_csv.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)

print("Modellverzeichnis:", models_dir.resolve())
print("\nVerfügbare Modelle:")
for f in sorted(models_dir.glob("*")):
    print(f"  - {f.name}")

## 2. Daten laden und vorbereiten

In [ ]:
# Splits laden
train_split = pd.read_csv(splits_csv / "train.csv")
val_split   = pd.read_csv(splits_csv / "val.csv")
test_split  = pd.read_csv(splits_csv / "test.csv")

print(f"Train: {len(train_split)} samples")
print(f"Val:   {len(val_split)} samples")
print(f"Test:  {len(test_split)} samples")
print(f"\nTotal: {len(train_split) + len(val_split) + len(test_split)} samples")

In [ ]:
# Label Encoder aufbauen
le = LabelEncoder()
le.fit(np.unique(np.concatenate([
    train_split["label"].values,
    val_split["label"].values,
    test_split["label"].values
])))

testY = le.transform(test_split["label"].values)
n_classes = len(le.classes_)

print(f"Klassen: {list(le.classes_)}")
print(f"Anzahl Klassen: {n_classes}")

In [ ]:
# Bilder laden (für klassische Modelle und CNN)
def load_images_from_splits(splits_df, img_size=(32, 32)):
    images, labels = [], []
    for _, row in tqdm(splits_df.iterrows(), total=len(splits_df), desc="Loading images"):
        img_path = pathlib.Path(row['path_processed']).resolve()
        if img_path.exists():
            try:
                img = Image.open(img_path).convert('L')
                img = img.resize(img_size, Image.BILINEAR)
                images.append(np.asarray(img, dtype=np.uint8))
                labels.append(row['label'])
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
        else:
            print(f"Missing: {img_path}")
    return np.array(images), np.array(labels)

print("Loading test images...")
testX, testY_labels = load_images_from_splits(test_split)
print(f"testX shape: {testX.shape}")

In [ ]:
# Für klassische Modelle: Flatten + Normalisieren + PCA
scaler = StandardScaler()

testX_flat = testX.reshape(-1, 32 * 32).astype('float32') / 255.0
testX_scaled = scaler.fit_transform(testX_flat)  # Fit auf Test ist OK, da nur für Vorhersage

pca = PCA(n_components=0.95)
testX_pca = pca.fit_transform(testX_scaled)  # Fit auf Test ist OK

print(f"PCA reduced dimensions: {testX_flat.shape[1]} -> {testX_pca.shape[1]}")

# Für CNN: In Kanäle umformen und normalisieren
testX_cnn = testX.reshape(-1, 32, 32, 1).astype('float32') / 255.0
print(f"CNN input shape: {testX_cnn.shape}")

## 3. Modelle laden und evaluieren

In [ ]:
# Dictionary für Modelle und Metadaten
models_info = {}

# === Decision Tree ===
print("\n" + "="*60)
print("DECISION TREE")
print("="*60)

with open(models_dir / "dt_best_model.pkl", "rb") as f:
    dt_model = pickle.load(f)
with open(models_dir / "dt_metadata.pkl", "rb") as f:
    dt_meta = pickle.load(f)

# Decision Tree wurde auf flachen 32x32-Pixeln (1024 Features) trainiert – also hier auch testX_flat nutzen
dt_pred = dt_model.predict(testX_flat)
dt_acc = accuracy_score(testY, dt_pred)

models_info['Decision Tree'] = {
    'model': dt_model,
    'metadata': dt_meta,
    'predictions': dt_pred,
    'accuracy': dt_acc,
    'data_used': 'Raw pixels (32x32, 1024 features)'
}

print(f"Model Type: {dt_meta['model_type']}")
print(f"Test Accuracy: {dt_acc:.4f}")
print(f"Tree Depth: {dt_meta['tree_depth']}")
print(f"Number of Nodes: {dt_meta['n_nodes']}")
print(f"Features Used: {testX_flat.shape[1]}")

In [ ]:
# === k-Nearest Neighbors ===
print("\n" + "="*60)
print("k-NEAREST NEIGHBORS")
print("="*60)

try:
    with open(models_dir / "knn_pca_model.pkl", "rb") as f:
        knn_model = pickle.load(f)
    with open(models_dir / "knn_metadata.pkl", "rb") as f:
        knn_meta = pickle.load(f)
except FileNotFoundError:
    print("WARN: k-NN Modell/Metadaten wurden nicht gefunden. Überspringe k-NN im Vergleich.")
else:
    # k-NN wurde auf PCA-Features aus knn_features.npz trainiert, nicht testX_pca
    knn_features_path = features_dir / "knn_features.npz"
    if not knn_features_path.exists():
        print("WARN: knn_features.npz wurde nicht gefunden. Überspringe k-NN.")
    else:
        knn_data = np.load(knn_features_path, allow_pickle=True)
        X_test_knn = knn_data["X_test"]
        y_test_knn = knn_data.get("y_test", testY)
        
        knn_pred = knn_model.predict(X_test_knn)
        knn_acc = accuracy_score(y_test_knn, knn_pred)

        models_info['k-NN'] = {
            'model': knn_model,
            'metadata': knn_meta,
            'predictions': knn_pred,
            'accuracy': knn_acc,
            'data_used': 'PCA Features (knn_features.npz)'
        }

        print(f"Model Type: {knn_meta['model_type']}")
        print(f"Test Accuracy: {knn_acc:.4f}")
        print(f"n_neighbors: {knn_meta['n_neighbors']}")
        print(f"Weights: {knn_meta['weights']}")
        print(f"Features: {knn_meta['n_features']}")

In [ ]:
# === Feedforward Neural Network ===
print("\n" + "="*60)
print("FEEDFORWARD NEURAL NETWORK (FFNN)")
print("="*60)

ffnn_features_path = features_dir / "ffnn_features.npz"
if not ffnn_features_path.exists():
    print("WARN: ffnn_features.npz wurde nicht gefunden. Überspringe FFNN.")
else:
    ffnn_data = np.load(ffnn_features_path, allow_pickle=True)
    X_test_ff = ffnn_data["X_test"]
    y_test_ff = ffnn_data.get("y_test", testY)  # fallback auf testY
    
    ffnn_model = load_model(models_dir / "ffnn_best_model.h5")
    with open(models_dir / "ffnn_metadata.pkl", "rb") as f:
        ffnn_meta = pickle.load(f)
    
    # Modell erwartet denselben Feature-Space wie beim Training (ffnn_features.npz)
    ffnn_pred_proba = ffnn_model.predict(X_test_ff, verbose=0)
    ffnn_pred = np.argmax(ffnn_pred_proba, axis=1)
    ffnn_acc = accuracy_score(y_test_ff, ffnn_pred)
    
    models_info['FFNN'] = {
        'model': ffnn_model,
        'metadata': ffnn_meta,
        'predictions': ffnn_pred,
        'accuracy': ffnn_acc,
        'data_used': 'Precomputed FFNN features (ffnn_features.npz)'
    }
    
    print(f"Model Type: {ffnn_meta['model_type']}")
    print(f"Test Accuracy: {ffnn_acc:.4f}")
    print(f"Regularization: {ffnn_meta['regularization']}")
    print(f"Input Features: {ffnn_meta['input_features']}")

In [ ]:
# === Convolutional Neural Network ===
print("\n" + "="*60)
print("CONVOLUTIONAL NEURAL NETWORK (CNN)")
print("="*60)

# CNN wurde auf 224x224 RGB-Bildern trainiert, nicht auf 32x32
# Bilder neu laden und resizen
def load_and_preprocess_for_cnn(splits_df, target_size=(224, 224)):
    images, labels = [], []
    for _, row in tqdm(splits_df.iterrows(), total=len(splits_df), desc="Loading CNN images"):
        img_path = pathlib.Path(row['path_processed']).resolve()
        if img_path.exists():
            try:
                img = Image.open(img_path).convert('RGB')  # CNN erwartet RGB
                img = img.resize(target_size, Image.BILINEAR)
                images.append(np.asarray(img, dtype=np.float32) / 255.0)
                labels.append(row['label'])
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
        else:
            print(f"Missing: {img_path}")
    return np.array(images), np.array(labels)

print("Loading test images for CNN (224x224 RGB)...")
testX_cnn_224, _ = load_and_preprocess_for_cnn(test_split)
print(f"CNN input shape: {testX_cnn_224.shape}")

cnn_model = load_model(models_dir / "cnn_best_model.h5")
with open(models_dir / "cnn_metadata.pkl", "rb") as f:
    cnn_meta = pickle.load(f)

cnn_pred_proba = cnn_model.predict(testX_cnn_224, verbose=0)
cnn_pred = np.argmax(cnn_pred_proba, axis=1)
cnn_acc = accuracy_score(testY, cnn_pred)

models_info['CNN'] = {
    'model': cnn_model,
    'metadata': cnn_meta,
    'predictions': cnn_pred,
    'accuracy': cnn_acc,
    'data_used': 'RGB Images (224x224)'
}

print(f"Model Type: {cnn_meta['model_type']}")
print(f"Test Accuracy: {cnn_acc:.4f}")
print(f"Regularization: {cnn_meta['regularization']}")
print(f"Input Shape: {cnn_meta['input_shape']}")

## 4. Vergleich: Überblick

In [ ]:
# Zusammenfassung
print("\n" + "="*70)
print("MODELLVERGLEICH - TEST ACCURACY")
print("="*70)

comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Test Accuracy': info['accuracy'],
        'Type': 'Classical ML' if name in ['Decision Tree', 'k-NN'] else 'Deep Learning',
        'Data Input': info['data_used'],
        'Data Type': 'Features (PCA)' if 'PCA' in info['data_used'] else 'Raw Images'
    }
    for name, info in models_info.items()
])

comparison_df = comparison_df.sort_values('Test Accuracy', ascending=False)
print(comparison_df.to_string(index=False))

# Bestes Modell
best_model_name = comparison_df.iloc[0]['Model']
best_model_acc = comparison_df.iloc[0]['Test Accuracy']

print(f"\n{'='*70}")
print(f"BEST MODEL: {best_model_name}")
print(f"TEST ACCURACY: {best_model_acc:.4f}")
print(f"{'='*70}")

## 5. Visualisierung: Vergleich der Genauigkeiten

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Balkendiagramm
model_names = list(models_info.keys())
accuracies = [models_info[name]['accuracy'] for name in model_names]
colors = ['#ff9999' if name != best_model_name else '#66bb6a' for name in model_names]

axes[0].bar(model_names, accuracies, color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Test Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('Modellvergleich: Test Accuracy', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].axhline(y=best_model_acc, color='green', linestyle='--', alpha=0.5, label='Best')
for i, (name, acc) in enumerate(zip(model_names, accuracies)):
    axes[0].text(i, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Ranking
ranking = comparison_df[['Model', 'Test Accuracy']].reset_index(drop=True)
ranking['Rank'] = ranking.index + 1
ranking_text = "\n".join([f"{row['Rank']}. {row['Model']}: {row['Test Accuracy']:.4f}" 
                           for _, row in ranking.iterrows()])

axes[1].text(0.1, 0.95, 'Ranking der Modelle', fontsize=13, fontweight='bold',
             transform=axes[1].transAxes, verticalalignment='top')
axes[1].text(0.1, 0.75, ranking_text, fontsize=11, family='monospace',
             transform=axes[1].transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[1].axis('off')

plt.tight_layout()
plt.show()

## 6. Detaillierte Analyse des besten Modells

In [ ]:
best_model_info = models_info[best_model_name]
best_predictions = best_model_info['predictions']
best_metadata = best_model_info['metadata']

print("\n" + "="*70)
print(f"DETAILLIERTE ANALYSE: {best_model_name.upper()}")
print("="*70)

print("\nModell-Eigenschaften:")
for key, value in best_metadata.items():
    print(f"  {key}: {value}")

print(f"\n✓ Test Accuracy: {best_model_acc:.4f} ({best_model_acc*100:.2f}%)")

### Classification Report (Best Model)

In [ ]:
print("\n" + "="*70)
print(f"CLASSIFICATION REPORT - {best_model_name}")
print("="*70)
print(classification_report(testY, best_predictions, target_names=le.classes_))

### Confusion Matrix (Best Model)

In [ ]:
cm = confusion_matrix(testY, best_predictions)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_,
            cbar_kws={'label': 'Count'}, linewidths=0.5)
plt.title(f'Confusion Matrix - {best_model_name}\n(Test Set)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=12, fontweight='bold')
plt.ylabel('True Class', fontsize=12, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Analyse per Klasse
print("\nPer-Klasse Metriken:")
for i, class_name in enumerate(le.classes_):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"  {class_name:15s} - Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")

## 7. Vergleich aller Modelle: Per-Klasse Performance

In [ ]:
# Pro Modell und Klasse: Recall berechnen
recall_per_class = {}

for model_name, model_info in models_info.items():
    preds = model_info['predictions']
    cm = confusion_matrix(testY, preds)
    
    recalls = []
    for i in range(n_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        recalls.append(recall)
    
    recall_per_class[model_name] = recalls

# Visualisierung
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(le.classes_))
width = 0.2

for i, (model_name, recalls) in enumerate(recall_per_class.items()):
    offset = (i - 1.5) * width
    color = '#66bb6a' if model_name == best_model_name else '#ff9999'
    ax.bar(x + offset, recalls, width, label=model_name, color=color, edgecolor='black', linewidth=0.8)

ax.set_xlabel('Klasse', fontsize=12, fontweight='bold')
ax.set_ylabel('Recall (True Positive Rate)', fontsize=12, fontweight='bold')
ax.set_title('Per-Klasse Recall Vergleich', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(le.classes_)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

## 8. Fehleranalyse des besten Modells

In [ ]:
# Falsch klassifizierte Samples finden
incorrect_mask = best_predictions != testY
n_incorrect = incorrect_mask.sum()
error_rate = n_incorrect / len(testY)

print(f"\nFehleranalyse für {best_model_name}:")
print(f"  Korrekt klassifiziert: {(~incorrect_mask).sum()} / {len(testY)}")
print(f"  Falsch klassifiziert: {n_incorrect} / {len(testY)}")
print(f"  Fehlerrate: {error_rate:.3f} ({error_rate*100:.1f}%)")

# Welche Klassen werden oft verwechselt?
cm_best = confusion_matrix(testY, best_predictions)

print(f"\nTop Verwechslungen (Off-Diagonal Einträge in Confusion Matrix):")
confusion_pairs = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j:
            count = cm_best[i, j]
            if count > 0:
                confusion_pairs.append((le.classes_[i], le.classes_[j], count))

confusion_pairs.sort(key=lambda x: x[2], reverse=True)
for true_class, pred_class, count in confusion_pairs[:5]:
    print(f"  {true_class:15s} → {pred_class:15s}: {count:3d} samples")

## 9. Zusammenfassung und Abschluss-Bericht

In [ ]:
print("\n" + "="*70)
print("MACHINE LEARNING PROJEKTABSCHLUSS - FINAL REPORT")
print("="*70)

print(f"\n DATENSATZ ÜBERSICHT")
print(f"-" * 70)
print(f"  Trainingssamples: {len(train_split)}")
print(f"  Validierungssamples: {len(val_split)}")
print(f"  Testsamples: {len(test_split)}")
print(f"  Anzahl Klassen: {n_classes}")
print(f"  Klassen: {', '.join(le.classes_)}")

print(f"\n MODELLE UND PERFORMANCE")
print(f"-" * 70)
for i, (model_name, info) in enumerate(comparison_df[['Model', 'Test Accuracy']].values, 1):
    marker = "Platz 1: " if i == 1 else "Platz 2: " if i == 2 else "Platz 3: " if i == 3 else f"  {i}"
    print(f"{marker} {model_name:20s}: {info:.4f} ({info*100:.2f}%)")

print(f"\n EMPFOHLENES MODELL: {best_model_name.upper()}")
print(f"-" * 70)
print(f"  Test Accuracy: {best_model_acc:.4f}")
print(f"  Fehlerrate: {error_rate:.4f} ({error_rate*100:.2f}%)")
print(f"  Korrekt klassifiziert: {(~incorrect_mask).sum()} von {len(testY)} Samples")

if best_model_name == 'Decision Tree':
    print(f"\n  Eigenschaften:")
    print(f"    - Baumtiefe: {best_metadata['tree_depth']}")
    print(f"    - Anzahl Knoten: {best_metadata['n_nodes']}")
    print(f"    - Regularisierung: Cost Complexity Pruning (α={best_metadata['best_alpha']:.6f})")
    print(f"\n  Vorteile:")
    print(f"    ✓ Sehr interpretierbar (whitebox model)")
    print(f"    ✓ Schnell zu trainieren und vorherzusagen")
    print(f"    ✓ Keine Skalierung/Normalisierung notwendig")
    print(f"    ✓ Feature importance verfügbar")
    print(f"\n  Nachteile:")
    print(f"    ✗ Unterliegt oft Overfitting ohne Pruning")
    print(f"    ✗ Instabil bei kleinen Datenveränderungen")
    
elif best_model_name == 'k-NN':
    print(f"\n  Eigenschaften:")
    print(f"    - n_neighbors: {best_metadata['n_neighbors']}")
    print(f"    - Gewichtung: {best_metadata['weights']}")
    print(f"    - Features: {best_metadata['n_features']}")
    print(f"\n  Vorteile:")
    print(f"    ✓ Einfach zu verstehen und implementieren")
    print(f"    ✓ Kein explizites Training erforderlich")
    print(f"    ✓ Mit PCA dimensionalitätsreduziert")
    print(f"\n  Nachteile:")
    print(f"    ✗ Rechenintensiv bei Vorhersage (alle Trainingssamples speichern)")
    print(f"    ✗ Empfindlich gegenüber Feature-Skalierung")
    
elif best_model_name == 'FFNN':
    print(f"\n  Eigenschaften:")
    print(f"    - Regularisierung: {best_metadata['regularization']}")
    print(f"    - Input Features: {best_metadata['input_features']}")
    print(f"\n  Vorteile:")
    print(f"    ✓ Lernt nichtlineare Entscheidungsgrenzen")
    print(f"    ✓ Gut regularisierbar (Dropout, L2)")
    print(f"    ✓ Scaliert zu größeren Datensätzen")
    print(f"\n  Nachteile:")
    print(f"    ✗ Blackbox-Modell, schwer interpretierbar")
    print(f"    ✗ Benötigt Hyperparameter-Tuning")
    
elif best_model_name == 'CNN':
    print(f"\n  Eigenschaften:")
    print(f"    - Input Shape: {best_metadata['input_shape']}")
    print(f"    - Regularisierung: {best_metadata['regularization']}")
    print(f"\n  Vorteile:")
    print(f"    ✓ Spezialisiert auf Bilddaten (räumliche Struktur)")
    print(f"    ✓ Lernt lokale Muster durch Convolutions")
    print(f"    ✓ Parameter-sharing reduziert Anzahl Parameter")
    print(f"    ✓ Translationsinvariant")
    print(f"\n  Nachteile:")
    print(f"    ✗ Rechenintensiv zu trainieren")
    print(f"    ✗ Benötigt mehr Daten als klassische ML")

print(f"\n EMPFEHLUNGEN FÜR DIE ZUKUNFT")
print(f"-" * 70)
print(f"  1. Ensemble-Methoden (Voting, Stacking) zur Verbesserung testen")
    
if 'CNN' not in [best_model_name]:
    print(f"  2. Transfer Learning (z.B. ResNet-Backbone) für CNN evaluieren")
    
print(f"  3. Hyperparameter-Optimization (Grid/Random Search) durchführen")
print(f"  4. Data Augmentation bei Limited Data")
print(f"  5. Class Imbalance Handling (Weighted Loss, SMOTE)")
print(f"  6. Cross-Validation für robustere Metriken")
print(f"  7. Explainability (SHAP, LIME) für Blackbox-Modelle")

print(f"="*70)